# Chain-of-Thought Distillation: SmolLM2-360M on GSM8K

Distill Claude Haiku 4.5's chain-of-thought reasoning into SmolLM2-360M-Instruct via SFT-LoRA, then measure accuracy and reasoning quality on GSM8K.

## Pipeline

1. Generate 1500 CoT solutions to GSM8K training problems with Claude Haiku 4.5.
2. Reject-sample: keep only CoTs whose final answer matches gold (~95% acceptance).
3. SFT-LoRA fine-tune SmolLM2-360M-Instruct on the filtered (question, CoT) pairs.
4. Evaluate baseline and fine-tuned models on 200 held-out problems.
5. Bootstrap confidence intervals on the per-problem difference.

Compute: single Kaggle T4 (free tier), ~$3 in Anthropic API costs.

The base model is small enough that distillation can transfer reasoning *style* but not arithmetic *capacity*. The qualitative analysis section is part of the result, not an afterthought.

## 1. Setup

Bitsandbytes is omitted: Kaggle's preinstalled `triton` is incompatible with `bitsandbytes==0.44.1`'s import path, and bf16 LoRA on a 360M model doesn't need 8-bit anyway. TRL is omitted in favor of `transformers.Trainer` + `peft` directly.

In [ ]:
!pip install -q -U transformers==4.45.2 peft==0.13.2 datasets==3.0.1 accelerate==1.0.1 anthropic
!pip uninstall -y bitsandbytes 2>/dev/null

In [ ]:
# Load secrets from Kaggle (do NOT hardcode keys)
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["ANTHROPIC_API_KEY"] = user_secrets.get_secret("ANTHROPIC_API_KEY")
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("✓ Auth ready")

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU not available — set Accelerator to GPU T4 x2 in Kaggle settings"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 2. Configuration

All hyperparameters live in this single cell — change one variable, re-run downstream cells, and the whole pipeline retrains on a different model or with different settings.

In [ ]:
# ---- Model and data ----
BASE_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
TEACHER_MODEL = "claude-haiku-4-5"
N_TRAIN_PROBLEMS = 1500           # GSM8K train problems to label with Claude
N_HELD_OUT = 200                  # GSM8K test problems for headline accuracy

# ---- LoRA ----
LORA_R = 16
LORA_ALPHA = 32
LORA_TARGET_MODULES = [           # Attention projections only — sufficient for 360M
    "q_proj", "k_proj", "v_proj", "o_proj",
]

# ---- Training ----
NUM_EPOCHS = 3
PER_DEVICE_BATCH = 4
GRAD_ACCUM_STEPS = 4              # effective batch = 16
LEARNING_RATE = 2e-4
MAX_SEQ_LEN = 1024

# ---- Eval ----
EVAL_BATCH_SIZE = 8
EVAL_MAX_NEW_TOKENS = 512

# ---- Reproducibility ----
SEED = 42

# ---- Paths ----
OUT_DIR = "/kaggle/working"
RAW_COTS_PATH = f"{OUT_DIR}/raw_cots.json"
FILTERED_COTS_PATH = f"{OUT_DIR}/filtered_cots.json"
ADAPTER_DIR = f"{OUT_DIR}/sft-final"

print(f"Base model:    {BASE_MODEL}")
print(f"Teacher model: {TEACHER_MODEL}")
print(f"Output dir:    {OUT_DIR}")

## 3. Helpers

Answer extraction, gold parsing, and unit-aware comparison. Used at three points in the pipeline:

- Filtering Claude's CoTs (does this CoT's answer match the gold?)
- Grading the baseline model
- Grading the fine-tuned model

Defining once and reusing means **all three phases use identical comparison logic**, which is essential for a clean before/after eval.

`answers_match` accepts numeric matches with tolerance for common unit conversions (cents↔dollars, minutes↔hours, etc.). This was added after empirical inspection revealed Claude occasionally reasons in cents while GSM8K's gold answer is in dollars — both correct, but naive string equality would mark them as different.

In [ ]:
import re

def _normalize(s):
    """Strip currency/units, normalize float-vs-int formatting."""
    s = s.replace(",", "").replace("$", "").strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(f)
    except ValueError:
        return s

def extract_answer(text):
    """Pull the final numeric answer from a CoT output. Tries multiple formats."""
    if text is None:
        return None
    # \boxed{X}
    m = re.search(r"\\boxed\{([^}]+)\}", text)
    if m: return _normalize(m.group(1))
    # #### X (GSM8K format)
    m = re.search(r"####\s*(-?[\d,]+\.?\d*)", text)
    if m: return _normalize(m.group(1))
    # "the answer is X"
    m = re.search(r"answer is\s*\$?(-?[\d,]+\.?\d*)", text, re.IGNORECASE)
    if m: return _normalize(m.group(1))
    # Last number in text
    nums = re.findall(r"-?[\d,]+\.?\d*", text)
    return _normalize(nums[-1]) if nums else None

def extract_gold(gsm8k_answer):
    """GSM8K's gold format ends with '#### <number>'."""
    return _normalize(gsm8k_answer.split("####")[-1].strip())

def answers_match(pred, gold):
    """Numeric comparison with tolerance for unit conversions."""
    if pred is None or gold is None:
        return False
    try:
        p, g = float(pred), float(gold)
    except (ValueError, TypeError):
        return str(pred).strip() == str(gold).strip()
    if abs(p - g) < 1e-4:
        return True
    # Common unit ratios: cents/dollars, minutes/hours, hours/days, etc.
    for ratio in [100, 60, 24, 12, 1000, 7, 52, 365]:
        if g != 0 and abs(p / g - ratio) < 1e-4: return True
        if p != 0 and abs(g / p - ratio) < 1e-4: return True
    return False

# Unit tests
assert extract_answer("...so #### 17") == "17"
assert extract_answer("\\boxed{3.5}") == "3.5"
assert answers_match("3", "300") is True   # dollars vs cents
assert answers_match("3.0", "3") is True   # float vs int
assert answers_match("4", "5") is False    # genuinely different
print("✓ Helpers ready")

## 4. Generate teacher CoT data

Generate 1500 CoT solutions from Claude Haiku 4.5 on GSM8K training problems. Filter via rejection sampling — keep only CoTs whose final answer matches gold.

**Skip behavior:** if `filtered_cots.json` already exists from a previous run, skip the whole API generation phase. The data is teacher-agnostic and reusable.

**Few-shot prompting**: the system prompt alone failed to prevent unit confusion (Claude reasoning in cents when GSM8K's gold is in dollars). Adding three few-shot demonstrations (one cents-to-dollars, one minutes-to-hours, one mixed) substantially reduced this failure mode. *Examples beat instructions* — a useful pattern for any LLM-in-the-loop pipeline.

**Cost:** ~$2.70 with Haiku 4.5 at $1/M input, $5/M output × 1500 calls × ~80 input + 250 output tokens. Generation takes ~45 min wall-clock at ~1.5s per call.

In [ ]:
import os
import json

if os.path.exists(FILTERED_COTS_PATH):
    with open(FILTERED_COTS_PATH) as f:
        filtered_cots = json.load(f)
    print(f"✓ Loaded {len(filtered_cots)} cached filtered CoTs from {FILTERED_COTS_PATH}")
    print("  Skipping API generation.")
    SKIP_GENERATION = True
else:
    print("No cached CoTs found. Will generate from Claude (~$3, ~45 min).")
    SKIP_GENERATION = False

In [ ]:
if not SKIP_GENERATION:
    from datasets import load_dataset
    import random

    gsm8k = load_dataset("openai/gsm8k", "main")
    random.seed(7)
    train_problems = list(gsm8k["train"])
    random.shuffle(train_problems)
    to_label = train_problems[:N_TRAIN_PROBLEMS]
    print(f"Will label {len(to_label)} problems")
else:
    print("Skipped (using cached CoTs)")

In [ ]:
if not SKIP_GENERATION:
    from anthropic import Anthropic
    import time
    from tqdm.auto import tqdm

    client = Anthropic()

    SYSTEM_PROMPT = """You are a math tutor. Solve the problem step by step.

Important reasoning guidelines:
- When a problem involves money, work in dollars throughout your reasoning, not cents.
  Convert cents to dollars early (e.g., "45 cents = $0.45") and keep all subsequent
  arithmetic in dollars.
- When a problem involves time, use the unit the question asks for (hours if it asks
  hours, minutes if minutes).
- The final answer should be in the most natural unit for the question.

End with the final answer in this exact format:
#### <number>

Use only the number itself after ####, with no units, no commas, no dollar signs."""

    # Few-shot examples — empirically these matter more than the system prompt
    FEW_SHOT = [
        {"role": "user", "content": "A pencil costs 30 cents. How much do 4 pencils cost?"},
        {"role": "assistant", "content": "I need to find the cost of 4 pencils.\n\nA pencil costs 30 cents, which is $0.30.\n\n4 pencils cost 4 × $0.30 = $1.20.\n\n#### 1.2"},
        {"role": "user", "content": "Two apples cost 50 cents. How much do 6 apples cost?"},
        {"role": "assistant", "content": "I need to find the cost of 6 apples.\n\nTwo apples cost 50 cents, which is $0.50. So one apple costs $0.50 / 2 = $0.25.\n\n6 apples cost 6 × $0.25 = $1.50.\n\n#### 1.5"},
        {"role": "user", "content": "A movie is 90 minutes long. How many hours is that?"},
        {"role": "assistant", "content": "I need to convert 90 minutes to hours.\n\nThere are 60 minutes in 1 hour.\n\n90 minutes ÷ 60 = 1.5 hours.\n\n#### 1.5"},
    ]

    def generate_cot(question, max_retries=3):
        messages = FEW_SHOT + [{"role": "user", "content": question}]
        for attempt in range(max_retries):
            try:
                resp = client.messages.create(
                    model=TEACHER_MODEL,
                    max_tokens=600,
                    system=SYSTEM_PROMPT,
                    messages=messages,
                    temperature=0.0,
                )
                return resp.content[0].text
            except Exception as e:
                print(f"Retry {attempt}: {e}")
                time.sleep(2 ** attempt)
        return None

    # Generate with checkpoint-every-100 in case Kaggle disconnects
    raw_cots = []
    for i, p in enumerate(tqdm(to_label, desc="Generating CoTs")):
        cot = generate_cot(p["question"])
        raw_cots.append({
            "question": p["question"],
            "gold": extract_gold(p["answer"]),
            "cot": cot,
        })
        if (i + 1) % 100 == 0:
            with open(RAW_COTS_PATH, "w") as f:
                json.dump(raw_cots, f)

    with open(RAW_COTS_PATH, "w") as f:
        json.dump(raw_cots, f)

    print(f"\nGenerated {len(raw_cots)} CoTs.")
    print(f"API failures: {sum(1 for x in raw_cots if x['cot'] is None)}")
else:
    print("Skipped")

In [ ]:
# Filter: rejection sampling — keep only CoTs whose answer matches gold
if not SKIP_GENERATION:
    filtered_cots = [
        x for x in raw_cots
        if x["cot"] is not None and answers_match(extract_answer(x["cot"]), x["gold"])
    ]

    with open(FILTERED_COTS_PATH, "w") as f:
        json.dump(filtered_cots, f)

    rate = len(filtered_cots) / len(raw_cots)
    print(f"Total CoTs:      {len(raw_cots)}")
    print(f"Correct CoTs:    {len(filtered_cots)}")
    print(f"Acceptance rate: {rate:.1%}")

print(f"\n✓ {len(filtered_cots)} filtered CoTs available for training")

## 5. Format data for SFT

Wrap each (question, CoT) pair in SmolLM2's chat template (ChatML-style with `<|im_start|>` / `<|im_end|>` tokens), then tokenize.

Critical detail: **training format must match inference format exactly**. The trained model will at eval time receive prompts wrapped in this same template, so any mismatch produces gibberish.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
import random

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def format_example(item):
    messages = [
        {"role": "user", "content": item["question"]},
        {"role": "assistant", "content": item["cot"]},
    ]
    return {"text": tok.apply_chat_template(messages, tokenize=False)}

formatted = [format_example(x) for x in filtered_cots]

# Inspect to verify chat template is correct
print("=" * 60)
print("Example formatted text (first 800 chars):")
print("=" * 60)
print(formatted[0]["text"][:800])
print("=" * 60)
print(f"\nFormatted {len(formatted)} examples")

In [ ]:
# Train/eval split — eval set is for monitoring training, not the headline metric
random.seed(SEED)
shuffled = formatted.copy()
random.shuffle(shuffled)

sft_eval_examples = shuffled[:50]
sft_train_examples = shuffled[50:]

# Tokenize for Trainer (padding handled per-batch by the collator)
def tokenize_fn(examples):
    out = tok(examples["text"], truncation=True, max_length=MAX_SEQ_LEN, padding=False)
    out["labels"] = [ids.copy() for ids in out["input_ids"]]
    return out

sft_train = Dataset.from_list(sft_train_examples).map(
    tokenize_fn, batched=True, remove_columns=["text"]
)
sft_eval = Dataset.from_list(sft_eval_examples).map(
    tokenize_fn, batched=True, remove_columns=["text"]
)

print(f"Train: {len(sft_train)} examples")
print(f"Eval:  {len(sft_eval)} examples")
print(f"Sample length: {len(sft_train[0]['input_ids'])} tokens")

## 6. Eval harness

Greedy generation (deterministic, reproducible) on a held-out test set. Same harness used for baseline and fine-tuned eval — apples-to-apples by construction.

Held-out set is sampled from GSM8K's `test` split (never seen during training). 200 problems is enough for tight bootstrap CIs (±~5pp at 95%) without making eval prohibitively slow.

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm

# Held-out test set
gsm8k = load_dataset("openai/gsm8k", "main")
random.seed(SEED)
test_problems = list(gsm8k["test"])
random.shuffle(test_problems)
held_out = test_problems[:N_HELD_OUT]
print(f"Held-out test set: {len(held_out)} problems")

# Left-padding required for batched causal generation
tok.padding_side = "left"

def evaluate(model, tokenizer, problems, max_new_tokens=EVAL_MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE):
    """Run greedy generation on each problem, score with answers_match."""
    correct = 0
    results = []
    for i in tqdm(range(0, len(problems), batch_size), desc="Evaluating"):
        batch = problems[i:i+batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": p["question"]}],
                tokenize=False, add_generation_prompt=True,
            )
            for p in batch
        ]
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_SEQ_LEN,
        ).to(model.device)

        with torch.no_grad():
            outs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        for j, p in enumerate(batch):
            input_len = inputs.input_ids.shape[1]
            generated = tokenizer.decode(outs[j][input_len:], skip_special_tokens=True)
            pred = extract_answer(generated)
            gold = extract_gold(p["answer"])
            is_correct = answers_match(pred, gold)
            correct += is_correct
            results.append({
                "question": p["question"], "gold": gold, "pred": pred,
                "correct": is_correct, "raw_output": generated,
            })
    return correct / len(problems), results

print("✓ Eval harness defined")

## 7. Baseline eval

Run the un-trained SmolLM2-360M on the held-out set. This is the "before" number.

Eval takes ~10 min on T4. Greedy decoding means this is fully deterministic — re-running gives identical results.

In [ ]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

print(f"Model loaded on {model.device}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
baseline_acc, baseline_results = evaluate(model, tok, held_out)

print(f"\n{'='*50}")
print(f"BASELINE ACCURACY: {baseline_acc:.3f} ({baseline_acc*100:.1f}%)")
print(f"{'='*50}")

with open(f"{OUT_DIR}/baseline_results.json", "w") as f:
    json.dump({
        "model": BASE_MODEL,
        "accuracy": baseline_acc,
        "n": len(held_out),
        "results": baseline_results,
    }, f)

# Sample outputs
print("\n--- Sample baseline outputs ---")
for r in baseline_results[:3]:
    status = "✓" if r["correct"] else "✗"
    print(f"\n{status} Gold: {r['gold']} | Pred: {r['pred']}")
    print(f"Output: {r['raw_output'][:250]}")

## 8. SFT-LoRA training

LoRA rank 16 over the four attention projections (q/k/v/o). Effective batch size 16 (per-device 4 × grad-accum 4). Cosine LR schedule with 5% warmup.

**Why LoRA over full fine-tuning at 360M scale?** Full fine-tuning of 360M weights is feasible on T4, but LoRA gives ~99.7% reduction in trainable parameters (~1M vs 360M) and ~4x speedup, with negligible quality difference for narrow tasks like this. It's also the more deployable artifact — a 4MB adapter file vs a 720MB full checkpoint.

Training takes ~20 minutes on T4.

In [ ]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model

# Right-padding for training (left-padding only needed for batched generation)
tok.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# DataCollatorForSeq2Seq pads input_ids AND labels properly for causal LM with custom labels
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tok, padding=True, return_tensors="pt", pad_to_multiple_of=8,
)

training_args = TrainingArguments(
    output_dir=f"{OUT_DIR}/sft-out",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    report_to="none",
    save_total_limit=2,
    remove_unused_columns=False,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=sft_train,
    eval_dataset=sft_eval,
    tokenizer=tok,
    data_collator=data_collator,
)

print("Starting training. ~20 min on T4.\n")
trainer.train()
print("\n✓ Training complete.")

trainer.save_model(ADAPTER_DIR)
print(f"✓ Adapter saved to {ADAPTER_DIR}")

## 9. Fine-tuned eval

Merge LoRA adapter back into base model weights for clean inference, then run the same eval harness.

In [ ]:
# Free training memory
del trainer, model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto",
)
ft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
ft_model = ft_model.merge_and_unload()
ft_model.eval()

# Switch back to left-padding for generation
tok.padding_side = "left"

print(f"Fine-tuned model on {ft_model.device}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
ft_acc, ft_results = evaluate(ft_model, tok, held_out)

# Compare to baseline
with open(f"{OUT_DIR}/baseline_results.json") as f:
    baseline = json.load(f)
baseline_acc = baseline["accuracy"]

print(f"\n{'='*50}")
print(f"BASELINE:    {baseline_acc:.3f} ({baseline_acc*100:.1f}%)")
print(f"FINE-TUNED:  {ft_acc:.3f} ({ft_acc*100:.1f}%)")
print(f"DELTA:       +{(ft_acc-baseline_acc)*100:.1f}pp ({ft_acc/max(baseline_acc, 1e-9):.2f}x)")
print(f"{'='*50}")

with open(f"{OUT_DIR}/ft_results.json", "w") as f:
    json.dump({
        "model": f"{BASE_MODEL} + LoRA SFT (distilled from {TEACHER_MODEL})",
        "accuracy": ft_acc,
        "n": len(held_out),
        "baseline_accuracy": baseline_acc,
        "improvement_pp": (ft_acc - baseline_acc) * 100,
        "results": ft_results,
    }, f)

print("\n--- Sample fine-tuned outputs ---")
for r in ft_results[:3]:
    status = "✓" if r["correct"] else "✗"
    print(f"\n{status} Gold: {r['gold']} | Pred: {r['pred']}")
    print(f"Output: {r['raw_output'][:250]}")

## 10. Bootstrap confidence intervals

Two analyses:

1. **Marginal CIs** — bootstrap baseline and fine-tuned accuracy independently to get 95% CIs. Tells us "is the headline number tight or noisy?"
2. **Paired bootstrap on the difference** — controls for the fact that some held-out problems are intrinsically harder than others. The right way to test "did fine-tuning help on a per-problem basis?"

The paired test is the more rigorous one. If its 95% CI excludes zero, the improvement is statistically significant.

In [ ]:
import numpy as np

def bootstrap_ci(results, n_boot=2000, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    correct = np.array([r["correct"] for r in results], dtype=int)
    n = len(correct)
    boots = np.array([correct[rng.integers(0, n, n)].mean() for _ in range(n_boot)])
    return np.percentile(boots, [100*alpha/2, 100*(1-alpha/2)])

baseline_ci = bootstrap_ci(baseline["results"])
ft_ci = bootstrap_ci(ft_results)

# Paired bootstrap on the per-problem difference
b_correct = np.array([r["correct"] for r in baseline["results"]], dtype=int)
f_correct = np.array([r["correct"] for r in ft_results], dtype=int)
diff = f_correct - b_correct
rng = np.random.default_rng(123)
diff_boots = np.array([diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(2000)])
diff_ci = np.percentile(diff_boots, [2.5, 97.5])
p_value = (diff_boots <= 0).mean()

print(f"Baseline:    {baseline['accuracy']*100:.1f}%  [95% CI: {baseline_ci[0]*100:.1f}, {baseline_ci[1]*100:.1f}]")
print(f"Fine-tuned:  {ft_acc*100:.1f}%  [95% CI: {ft_ci[0]*100:.1f}, {ft_ci[1]*100:.1f}]")
print(f"\nPaired difference: +{diff.mean()*100:.1f}pp  [95% CI: {diff_ci[0]*100:.1f}, {diff_ci[1]*100:.1f}]")
print(f"P(fine-tuned ≤ baseline) ≈ {p_value:.3f}")

with open(f"{OUT_DIR}/stats.json", "w") as f:
    json.dump({
        "baseline_acc": baseline["accuracy"], "baseline_ci": baseline_ci.tolist(),
        "ft_acc": ft_acc, "ft_ci": ft_ci.tolist(),
        "diff_mean": float(diff.mean()), "diff_ci": diff_ci.tolist(),
        "p_value": float(p_value),
    }, f)

## 11. Qualitative analysis: style transfer vs. capability transfer

Headline accuracy is one measurement of distillation success. Equally important: **did the trained model adopt the teacher's reasoning structure?**

This section pairs baseline and fine-tuned outputs on the same problems and inspects the differences. Distillation at this scale is best understood as **style transfer** (the student adopts the teacher's reasoning *format*) rather than **capability transfer** (the student becomes capable of harder math).

In [ ]:
# Side-by-side comparison: baseline vs. fine-tuned on the same problems
print("=" * 70)
print("BASELINE vs. FINE-TUNED — same problems, same eval harness")
print("=" * 70)

# Show pairs where ft model improved AND pairs where it didn't, for honesty
improved = [i for i in range(len(ft_results)) if ft_results[i]["correct"] and not baseline["results"][i]["correct"]]
regressed = [i for i in range(len(ft_results)) if baseline["results"][i]["correct"] and not ft_results[i]["correct"]]

print(f"\nProblems where fine-tuned improved over baseline: {len(improved)}")
print(f"Problems where fine-tuned regressed below baseline: {len(regressed)}")
print(f"Net improvement: {len(improved) - len(regressed)} problems\n")

# Show 2 wins
print("\n" + "─" * 70)
print("TWO WINS (baseline wrong → fine-tuned correct)")
print("─" * 70)
for idx in improved[:2]:
    b = baseline["results"][idx]
    f = ft_results[idx]
    print(f"\nQuestion: {b['question'][:200]}")
    print(f"Gold: {b['gold']}")
    print(f"\n  BASELINE  (✗ pred={b['pred']}): {b['raw_output'][:300]}...")
    print(f"\n  FINE-TUNED (✓ pred={f['pred']}): {f['raw_output'][:300]}...")
    print()

# Show 1 representative case where both got it wrong but reasoning quality differs
both_wrong = [i for i in range(len(ft_results))
              if not baseline["results"][i]["correct"] and not ft_results[i]["correct"]]

if both_wrong:
    print("\n" + "─" * 70)
    print("BOTH WRONG, but compare the reasoning quality")
    print("─" * 70)
    idx = both_wrong[0]
    b = baseline["results"][idx]
    f = ft_results[idx]
    print(f"\nQuestion: {b['question'][:200]}")
    print(f"Gold: {b['gold']}")
    print(f"\n  BASELINE  (pred={b['pred']}): {b['raw_output'][:350]}...")
    print(f"\n  FINE-TUNED (pred={f['pred']}): {f['raw_output'][:350]}...")

In [ ]:
# Generation-length statistics: did distillation change verbosity?
import statistics

baseline_lens = [len(r["raw_output"]) for r in baseline["results"]]
ft_lens = [len(r["raw_output"]) for r in ft_results]

# Did the fine-tuned model learn to use the #### marker?
baseline_with_marker = sum(1 for r in baseline["results"] if "####" in r["raw_output"])
ft_with_marker = sum(1 for r in ft_results if "####" in r["raw_output"])

print("Generation length (chars):")
print(f"  Baseline:   median={statistics.median(baseline_lens):.0f}, mean={statistics.mean(baseline_lens):.0f}")
print(f"  Fine-tuned: median={statistics.median(ft_lens):.0f}, mean={statistics.mean(ft_lens):.0f}")

print(f"\nFinal-answer marker (####) presence:")
print(f"  Baseline:   {baseline_with_marker}/{len(baseline['results'])} ({baseline_with_marker/len(baseline['results']):.1%})")
print(f"  Fine-tuned: {ft_with_marker}/{len(ft_results)} ({ft_with_marker/len(ft_results):.1%})")

## 12. Training loss curve

Training and eval loss across the 3 epochs. Healthy training shows monotonically decreasing train loss with eval loss tracking closely (not diverging — that would indicate overfitting).

In [ ]:
import matplotlib.pyplot as plt
import os

# Reload trainer state from latest checkpoint (it has the full log_history)
ckpts = sorted([d for d in os.listdir(f"{OUT_DIR}/sft-out") if d.startswith("checkpoint-")])
with open(f"{OUT_DIR}/sft-out/{ckpts[-1]}/trainer_state.json") as f:
    state = json.load(f)

train_loss = [(e["step"], e["loss"]) for e in state["log_history"] if "loss" in e]
eval_loss = [(e["step"], e["eval_loss"]) for e in state["log_history"] if "eval_loss" in e]

fig, ax = plt.subplots(figsize=(8, 5))
if train_loss:
    ax.plot([s for s,_ in train_loss], [l for _,l in train_loss], label="Train", alpha=0.7)
if eval_loss:
    ax.plot([s for s,_ in eval_loss], [l for _,l in eval_loss],
            label="Eval", marker="o", linewidth=2)
ax.set_xlabel("Step"); ax.set_ylabel("Loss"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title(f"SFT loss: {BASE_MODEL.split('/')[-1]} distilled from {TEACHER_MODEL}")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/loss_curve.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"✓ Saved {OUT_DIR}/loss_curve.png")

## 13. Push to HuggingFace Hub

Make the trained adapter publicly accessible. The HF link goes in the README and resume.

In [ ]:
HF_REPO_NAME = "smollm2-360m-gsm8k-distilled-haiku"   # rename as desired

ft_model.push_to_hub(HF_REPO_NAME)
tok.push_to_hub(HF_REPO_NAME)
print(f"✓ Pushed model and tokenizer to https://huggingface.co/<your-username>/{HF_REPO_NAME}")

## 14. Results summary

This cell loads all results files and prints a final summary block — easy to screenshot for your README, or to share as a self-contained eval report.

In [ ]:
import json

with open(f"{OUT_DIR}/baseline_results.json") as f:
    baseline = json.load(f)
with open(f"{OUT_DIR}/ft_results.json") as f:
    ft = json.load(f)
with open(f"{OUT_DIR}/stats.json") as f:
    stats = json.load(f)

print("=" * 60)
print(f"  CoT Distillation: {BASE_MODEL.split('/')[-1]}")
print(f"  Teacher: {TEACHER_MODEL}")
print("=" * 60)
print(f"  Training data:    {len(filtered_cots)} CoTs (from {N_TRAIN_PROBLEMS} generated)")
print(f"  Held-out test:    {N_HELD_OUT} GSM8K problems")
print(f"  LoRA config:      r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  Training:         {NUM_EPOCHS} epochs, lr={LEARNING_RATE}, eff. batch=16")
print("=" * 60)
print(f"  Baseline:    {stats['baseline_acc']*100:.1f}%  "
      f"[95% CI: {stats['baseline_ci'][0]*100:.1f}–{stats['baseline_ci'][1]*100:.1f}]")
print(f"  Fine-tuned:  {stats['ft_acc']*100:.1f}%  "
      f"[95% CI: {stats['ft_ci'][0]*100:.1f}–{stats['ft_ci'][1]*100:.1f}]")
print(f"  Δ (paired):  +{stats['diff_mean']*100:.1f}pp  "
      f"[95% CI: {stats['diff_ci'][0]*100:.1f}–{stats['diff_ci'][1]*100:.1f}]")
print(f"  P(no improvement) ≈ {stats['p_value']:.3f}")
print("=" * 60)